# General

For more informations, si the documentation *Documentary Strategy*.

# Import & Configs

In [1]:
from Bio import Entrez
import pandas as pd
from tqdm import tqdm
import json

from pathlib import Path
import tomllib

In [2]:
def get_secrets():
    secrets_path = Path("../.secrets.toml")
    
    with open(secrets_path, "rb") as f:
        secrets = tomllib.load(f)
        
    return secrets

secrets = get_secrets()

In [3]:
Entrez.email = secrets["PUBMED_EMAIL"]

# PubMed extraction

PubMed query:

In [4]:
QUERY = """
(
    glioma
    OR glioblastoma
    OR meningioma
    OR pituitary tumor
)
AND
(
    MRI
    OR brain MRI
    OR radiology
    OR neuroimaging
)
"""

Get the pubmed id :

In [5]:
handle = Entrez.esearch(
    db="pubmed",
    term=QUERY,
    retmax=500
)

record = Entrez.read(handle)

pmids = record["IdList"]

len(pmids)

500

Download abstracts:

In [6]:
papers = []

for pmid in tqdm(pmids):

    try:
        handle = Entrez.efetch(
            db="pubmed",
            id=pmid,
            rettype="medline",
            retmode="text"
        )

        text = handle.read()

        papers.append({
            "pmid": pmid,
            "raw_text": text
        })

    except Exception as e:
        print(f"Error with {pmid}: {e}")

100%|██████████| 500/500 [03:27<00:00,  2.41it/s]


Save data as JSON Line:

In [8]:
output_path = "../data/raw/pubmed_abstracts.jsonl"

with open(output_path, "w") as f:

    for paper in papers:
        f.write(json.dumps(paper) + "\n")

Dataset loaded and saved.